# 🧬 Group A — Notebook 1: Understand and Fine-Tune DNABERT

## Main question

> **Can a pretrained DNA language model distinguish CTCF-binding DNA from background DNA?**

This notebook assumes **no previous Python or machine-learning experience**.

We will still use real Python, PyTorch, pandas, and model-evaluation tools—but each tool appears only when it helps answer the scientific question.

## How to read this notebook

| Marker | Meaning |
|---|---|
| 📖 **IDEA** | Understand the concept. |
| ▶️ **RUN** | Run the short code cell. |
| 👀 **READ** | This code expresses an important model idea. |
| ✏️ **CHANGE** | Change one value and test your prediction. |
| 🔒 **HELPER** | Working machinery. You may open it, but you do not need to study it. |
| ✅ **CHECKPOINT** | Explain the idea in your own words. |

The goal is **not** to memorize Python syntax. The goal is to understand what each part of the model is doing.

## Roadmap

```mermaid
flowchart LR
    A["Biological question"] --> B["Look at DNA data"]
    B --> C["DNA → 6-mers"]
    C --> D["Token IDs"]
    D --> E["Pretrained DNABERT"]
    E --> F["Classification head"]
    F --> G["Fine-tune"]
    G --> H["Evaluate with graphs"]
```

# 1. The biological problem

DNA is made from four letters:

```text
A   C   G   T
```

**CTCF** is a protein that can bind particular DNA regions.

We turn the biology into a two-class prediction problem:

```mermaid
flowchart LR
    A["DNA sequence"] --> B["Model"]
    B --> C{"Prediction"}
    C -->|"1"| D["CTCF Binding"]
    C -->|"0"| E["Background"]
```

The model never sees the protein itself. It learns sequence patterns associated with the two labels.

### Where do the positive examples come from?

A simplified ChIP-seq idea is:

```mermaid
flowchart TD
    A["Cells"] --> B["CTCF is attached to DNA"]
    B --> C["Capture DNA associated with CTCF"]
    C --> D["Sequence those DNA fragments"]
    D --> E["CTCF-associated genomic regions"]
    E --> F["Positive training examples"]
```

For this bootcamp, the dataset has already been prepared on Perlmutter.

In [ ]:
# ▶️ RUN — imports and paths

from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from transformers import AutoTokenizer, BertModel

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

DATA_DIR = Path("/global/cfs/cdirs/m4388/projects/project7/ctcf_k562_example")

# Local DNABERT copy on Perlmutter.
# No Hugging Face download is needed.
MODEL_PATH = Path("/global/cfs/cdirs/m4388/projects/project7/models/DNA_bert_6")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dataset:", DATA_DIR)
print("DNABERT:", MODEL_PATH)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

### What are those imports doing?

We are **not studying the libraries themselves**. We use them as tools:

- `pandas` → makes small tables easy to read.
- `matplotlib` → draws graphs.
- `torch` → trains neural networks.
- `transformers` → loads DNABERT from the local folder.
- `sklearn.metrics` → provides standard evaluation calculations.

Think of an import as:

> “Python, please give me this toolbox.”

In [ ]:
import random
import numpy as np

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


def load_dataset(data_dir):
    """Read DNA + labels, remove invalid/conflicting duplicates."""
    seq_path = Path(data_dir) / "seqs.txt"
    label_path = Path(data_dir) / "labels.txt"

    sequences = [
        line.strip().upper()
        for line in seq_path.read_text().splitlines()
        if line.strip()
    ]

    labels = [
        int(line.strip())
        for line in label_path.read_text().splitlines()
        if line.strip()
    ]

    if len(sequences) != len(labels):
        raise ValueError("Sequence and label counts do not match.")

    label_sets = {}

    for sequence, label in zip(sequences, labels):
        if set(sequence) <= set("ACGT"):
            label_sets.setdefault(sequence, set()).add(label)

    clean_sequences = [
        sequence
        for sequence, values in label_sets.items()
        if len(values) == 1
    ]

    clean_labels = [
        next(iter(label_sets[sequence]))
        for sequence in clean_sequences
    ]

    return clean_sequences, clean_labels


def to_6mer_text(sequence, k=6):
    """Convert one DNA string into overlapping k-mer text."""
    return " ".join(
        sequence[i:i+k]
        for i in range(len(sequence) - k + 1)
    )


class DNABertDataset(Dataset):
    """Prepare token IDs, attention masks, and labels for PyTorch."""

    def __init__(self, sequences, labels, tokenizer, max_length=256):
        dna_as_words = [
            to_6mer_text(sequence)
            for sequence in sequences
        ]

        encoded = tokenizer(
            dna_as_words,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        self.input_ids = encoded["input_ids"]
        self.attention_mask = encoded["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return (
            self.input_ids[index],
            self.attention_mask[index],
            self.labels[index],
        )


def split_dataset(sequences, labels, seed=42):
    """Use 80% for training and 20% for validation."""
    return train_test_split(
        sequences,
        labels,
        test_size=0.20,
        random_state=seed,
        stratify=labels,
    )


def train_dnabert(
    model,
    tokenizer,
    sequences,
    labels,
    epochs=2,
    batch_size=8,
    learning_rate=1e-4,
    max_length=256,
    seed=42,
):
    """Fine-tune DNABERT and return history, metrics, and predictions."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_seq, val_seq, train_y, val_y = split_dataset(
        sequences,
        labels,
        seed=seed,
    )

    train_data = DNABertDataset(
        train_seq,
        train_y,
        tokenizer,
        max_length=max_length,
    )

    val_data = DNABertDataset(
        val_seq,
        val_y,
        tokenizer,
        max_length=max_length,
    )

    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_data,
        batch_size=batch_size,
        shuffle=False,
    )

    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
    )

    loss_function = nn.CrossEntropyLoss()

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auroc": [],
        "val_auprc": [],
    }

    for epoch in range(1, epochs + 1):

        # -------- training --------
        model.train()
        training_loss_sum = 0.0

        for input_ids, attention_mask, answers in train_loader:
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            answers = answers.to(DEVICE)

            optimizer.zero_grad()

            prediction_scores = model(
                input_ids,
                attention_mask,
            )

            loss = loss_function(
                prediction_scores,
                answers,
            )

            loss.backward()
            optimizer.step()

            training_loss_sum += loss.item() * len(answers)

        training_loss = training_loss_sum / len(train_data)

        # -------- validation --------
        model.eval()

        validation_loss_sum = 0.0
        true_labels = []
        predicted_labels = []
        binding_probabilities = []

        with torch.no_grad():
            for input_ids, attention_mask, answers in val_loader:
                input_ids = input_ids.to(DEVICE)
                attention_mask = attention_mask.to(DEVICE)
                answers = answers.to(DEVICE)

                prediction_scores = model(
                    input_ids,
                    attention_mask,
                )

                loss = loss_function(
                    prediction_scores,
                    answers,
                )

                probabilities = torch.softmax(
                    prediction_scores,
                    dim=1,
                )[:, 1]

                predictions = prediction_scores.argmax(dim=1)

                validation_loss_sum += loss.item() * len(answers)

                true_labels.extend(answers.cpu().tolist())
                predicted_labels.extend(predictions.cpu().tolist())
                binding_probabilities.extend(probabilities.cpu().tolist())

        validation_loss = validation_loss_sum / len(val_data)

        auroc = roc_auc_score(
            true_labels,
            binding_probabilities,
        )

        auprc = average_precision_score(
            true_labels,
            binding_probabilities,
        )

        history["train_loss"].append(training_loss)
        history["val_loss"].append(validation_loss)
        history["val_auroc"].append(auroc)
        history["val_auprc"].append(auprc)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"train loss={training_loss:.4f} | "
            f"val loss={validation_loss:.4f} | "
            f"AUROC={auroc:.4f}"
        )

    metrics = {
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels,
        ),
        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "f1": f1_score(
            true_labels,
            predicted_labels,
            zero_division=0,
        ),
        "auroc": roc_auc_score(
            true_labels,
            binding_probabilities,
        ),
        "auprc": average_precision_score(
            true_labels,
            binding_probabilities,
        ),
    }

    evaluation = {
        "true": true_labels,
        "predicted": predicted_labels,
        "probability": binding_probabilities,
    }

    return model, history, metrics, evaluation

# 2. Look at the data

The helper function:

```python
load_dataset(DATA_DIR)
```

does three things:

1. reads `seqs.txt` and `labels.txt`,
2. removes invalid DNA and ambiguous duplicates,
3. returns two ordinary Python lists.

We then use pandas **only to display those lists as a convenient table**.

In [ ]:
# ▶️ RUN — load DNA, then display it as a table

sequences, labels = load_dataset(DATA_DIR)

data = pd.DataFrame({
    "sequence": sequences,
    "label": labels,
})

data["label_name"] = data["label"].map({
    0: "Background",
    1: "Binding",
})

print("Number of examples:", len(data))

data.head()

### Reading the pandas code

```python
pd.DataFrame(...)
```

simply means:

> “Put these lists into rows and columns so humans can read them easily.”

`data.head()` means:

> “Show the first five rows.”

That is enough pandas knowledge for this section.

In [ ]:
# ▶️ RUN — class balance

class_counts = data["label_name"].value_counts()

class_counts.plot(kind="bar")

plt.ylabel("Number of DNA sequences")
plt.title("How many examples are in each class?")
plt.show()

### How to read this graph

The x-axis shows the two classes.  
The y-axis shows how many examples belong to each class.

**Question this graph answers:**

> Is one class much more common than the other?

Large class imbalance can make accuracy misleading, so we check this before training.

In [ ]:
# ▶️ RUN — sequence length

data["length"] = data["sequence"].str.len()

data["length"].plot(kind="hist", bins=20)

plt.xlabel("DNA sequence length (bases)")
plt.title("Are the DNA sequences similar in length?")
plt.show()

### Why inspect sequence length?

A model should not accidentally learn something trivial like:

> “Binding sequences are longer.”

Here we use the graph as a quick quality-control check, not as a pandas lesson.

# 3. DNA becomes 6-mers

DNABERT-6 does not read one nucleotide at a time.

It reads overlapping DNA “words” that are six bases long:

```mermaid
flowchart LR
    A["Raw DNA"] --> B["Overlapping 6-mers"]
    B --> C["Vocabulary lookup"]
    C --> D["Token IDs"]
    D --> E["Embedding vectors"]
    E --> F["DNABERT"]
```

Example:

```text
ACGTACGT

ACGTAC
 CGTACG
  GTACGT
```

### Function: `to_6mer_text(sequence)`

This helper receives one DNA string and returns overlapping 6-mers separated by spaces.

The important parameter is:

- `k=6` → each DNA word contains six bases.

In [ ]:
# ▶️ RUN — see one sequence become 6-mers

example_dna = sequences[0]

six_mer_text = to_6mer_text(example_dna)
six_mers = six_mer_text.split()

print("First 30 bases:")
print(example_dna[:30])

print()
print("First five 6-mers:")
print(six_mers[:5])

print()
print("Total 6-mers:", len(six_mers))

# 4. Token → token ID → embedding

These are three different things:

```mermaid
flowchart LR
    A["6-mer<br/>ACGTGC"] --> B["Token ID<br/>integer"]
    B --> C["Embedding lookup"]
    C --> D["Vector of numbers"]
```

Neural networks work with numbers, so the tokenizer connects DNA words to integer IDs.

### Function: `AutoTokenizer.from_pretrained(...)`

Here **“from_pretrained” does not mean download**.

We point it to the local folder:

```text
/global/cfs/cdirs/m4388/projects/project7/models/DNA_bert_6
```

`local_files_only=True` tells the library:

> Use only files already present on Perlmutter.

In [ ]:
# ▶️ RUN — load the local DNABERT tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
)

tokens = tokenizer.tokenize(six_mer_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("First tokens:", tokens[:8])
print("First token IDs:", token_ids[:8])

# 5. What is inside DNABERT?

DNABERT is a Transformer.

A simplified attention idea is:

- **Query (Q):** What information am I looking for?
- **Key (K):** What information do I contain?
- **Value (V):** What information should I pass forward?

```mermaid
flowchart TD
    A["Token representation"] --> B["Query Q"]
    A --> C["Key K"]
    A --> D["Value V"]
    B --> E["Compare Query with Keys"]
    C --> E
    E --> F["Attention weights"]
    D --> G["Weighted information"]
    F --> G
```

You do **not** have to implement Q/K/V in Group A. DNABERT already contains those Transformer layers.

# 6. Add a classifier to DNABERT

DNABERT produces a contextual representation for every token.

For classification we use the final `[CLS]` representation as a sequence summary:

```mermaid
flowchart TD
    A["Tokenized DNA"] --> B["Pretrained DNABERT"]
    B --> C["Final [CLS] vector"]
    C --> D["Linear layer"]
    D --> E["2 scores"]
    E --> F["Background / Binding"]
```

## Read the next class by following its parts

**`nn.Module`**  
PyTorch's base class for neural-network models.

**`__init__`**  
Builds the model's parts.

**`BertModel.from_pretrained(...)`**  
Loads the pretrained DNABERT layers from the local folder.

**`nn.Linear(hidden_size, 2)`**  
Adds a small layer that converts the DNABERT summary into two class scores.

**`forward(...)`**  
Describes the path data follows through the model.

In [ ]:
# 👀 READ — the important DNABERT model code

class DNABertClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        # Part 1: pretrained DNA Transformer
        self.dnabert = BertModel.from_pretrained(
            str(MODEL_PATH),
            local_files_only=True,
            add_pooling_layer=False,
        )

        hidden_size = self.dnabert.config.hidden_size

        # Part 2: our two-class prediction layer
        self.classifier = nn.Linear(
            hidden_size,
            2,
        )

    def forward(self, input_ids, attention_mask):

        # DNA token IDs move through DNABERT.
        output = self.dnabert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Use [CLS] as a summary of the complete sequence.
        dna_summary = output.last_hidden_state[:, 0, :]

        # Produce one score for each class.
        return self.classifier(dna_summary)

In [ ]:
# ▶️ RUN — build one model

model = DNABertClassifier()

print("Transformer layers:", model.dnabert.config.num_hidden_layers)
print("Hidden size:", model.dnabert.config.hidden_size)
print("Attention heads:", model.dnabert.config.num_attention_heads)

# 7. What happens during training?

```mermaid
flowchart LR
    A["Batch of DNA"] --> B["Model predicts"]
    B --> C["Calculate loss"]
    C --> D["Backpropagation"]
    D --> E["Optimizer updates parameters"]
    E --> A
```

The five important PyTorch ideas are:

```python
optimizer.zero_grad()
prediction = model(...)
loss = loss_function(prediction, answer)
loss.backward()
optimizer.step()
```

The full loop is hidden inside `train_dnabert(...)` because batching, CUDA movement, and bookkeeping are not the main lesson.

## Function: `train_dnabert(...)`

You give this function:

- `model` → the DNABERT classifier,
- `tokenizer` → converts DNA into token IDs,
- `sequences` and `labels` → the dataset,
- `epochs` → how many times to pass through the training set,
- `batch_size` → how many examples are processed together,
- `learning_rate` → how large parameter updates are.

It returns:

- the trained model,
- a history of learning,
- final metrics,
- validation predictions for graphs.

In [ ]:
# ✏️ CHANGE — simple training settings

EPOCHS = 2
BATCH_SIZE = 8
LEARNING_RATE = 1e-4

In [ ]:
# ▶️ RUN — fine-tune DNABERT

model, history, metrics, evaluation = train_dnabert(
    model=model,
    tokenizer=tokenizer,
    sequences=sequences,
    labels=labels,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

# 8. Read the model result

We use pandas again only because a small table is easier to scan than many separate `print()` statements.

In [ ]:
# ▶️ RUN — one-row metrics table

results_table = pd.DataFrame([
    metrics
])

results_table.round(3)

### What do the metrics mean?

- **Accuracy:** fraction of all predictions that were correct.
- **Precision:** when the model says “Binding,” how often is it right?
- **Recall:** of the real binding sequences, how many did it find?
- **F1:** balances precision and recall.
- **AUROC:** how well binding sequences are ranked above background across thresholds.
- **AUPRC:** precision–recall performance across thresholds.

No single metric tells the complete story, so we also use graphs.

## Graph 1 — Training and validation loss

**Loss** is the model's error signal during training.

Usually we hope to see training loss decrease.

Validation loss tells us whether improvement also transfers to data the model did not train on.

In [ ]:
# ▶️ RUN — loss curves

epoch_numbers = range(
    1,
    len(history["train_loss"]) + 1,
)

plt.plot(
    epoch_numbers,
    history["train_loss"],
    marker="o",
    label="Training loss",
)

plt.plot(
    epoch_numbers,
    history["val_loss"],
    marker="o",
    label="Validation loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("How did error change during training?")
plt.legend()
plt.show()

**How to interpret it**

- Training loss going down → the model is fitting the training data.
- Validation loss also going down → learning may generalize.
- Training loss falling while validation loss rises → possible overfitting.

## Graph 2 — Confusion matrix

A confusion matrix counts **what kind of mistakes** the model made.

```text
                Predicted
             Background  Binding
Actual
Background       TN        FP
Binding          FN        TP
```

- **FP:** background incorrectly called binding.
- **FN:** binding incorrectly called background.

In [ ]:
# ▶️ RUN — confusion matrix

cm = confusion_matrix(
    evaluation["true"],
    evaluation["predicted"],
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Background", "Binding"],
).plot()

plt.title("What kinds of mistakes did DNABERT make?")
plt.show()

**What looks good?**

We want most counts on the diagonal:

```text
Background → Background
Binding    → Binding
```

The off-diagonal cells are mistakes.

## Graph 3 — ROC curve

The ROC curve asks:

> As we change the prediction threshold, how well can the model separate the two classes?

A random classifier follows roughly the diagonal. Better models bend toward the upper-left.

In [ ]:
# ▶️ RUN — ROC curve

false_positive_rate, true_positive_rate, _ = roc_curve(
    evaluation["true"],
    evaluation["probability"],
)

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"AUROC = {metrics['auroc']:.3f}",
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve")
plt.legend()
plt.show()

## Graph 4 — Precision–Recall curve

This graph focuses directly on the tradeoff between:

- **precision:** how trustworthy positive predictions are;
- **recall:** how many true positives are found.

It is especially useful when classes are imbalanced.

In [ ]:
# ▶️ RUN — precision–recall curve

precision, recall, _ = precision_recall_curve(
    evaluation["true"],
    evaluation["probability"],
)

plt.plot(
    recall,
    precision,
    label=f"AUPRC = {metrics['auprc']:.3f}",
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve")
plt.legend()
plt.show()

# 9. One controlled experiment

Change **one** training setting.

For example:

```python
LEARNING_RATE = 2e-5
```

or:

```python
EPOCHS = 4
```

Before running, write one prediction:

> I think this change will ______ because ______.

Controlled changes make comparisons easier to interpret.

# ✅ Group A summary

```mermaid
flowchart LR
    A["DNA dataset"] --> B["Quick inspection"]
    B --> C["6-mer tokenizer"]
    C --> D["Local pretrained DNABERT"]
    D --> E["Classification head"]
    E --> F["Fine-tuning"]
    F --> G["Loss"]
    F --> H["Confusion matrix"]
    F --> I["ROC / PR curves"]
```

You are ready for Notebook 3A if you can explain the pipeline without code.